# Read YOLO Dataset Annotations to DataFrame

This notebook reads all YOLO annotations from train, val, holdout, and holdout_v2 splits,
extracts camera height from filenames, and saves to CSV.

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
# Dataset path
DATASET_PATH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_10_margin_15_bg_wb_alif_clahe/labels"
OUTPUT_CSV = "yolo_annotations.csv"

# Subsets to process
SUBSETS = ["train", "val", "holdout", "holdout_v2"]

In [ ]:
def extract_camera_height(filename: str) -> int | None:
    """
    Extract camera height from filename and round to nearest 10 cm.

    Examples:
    - 4.2_batch1____4.2_H260_menivim_5.txt -> 260
    - 4.2_batch1____4.2_H277_menivim_OS_28.txt -> 280
    - 4.2_batch2____4.2_H_cam_234_spacesbooth_100.txt -> 230
    - 4.2_H289_file.txt -> 290
    - 4.2_H272_file.txt -> 270

    Args:
        filename: Name of the annotation file

    Returns:
        Camera height in cm (rounded to nearest 10), or None if not found
    """
    # Try pattern: H<number> or H_cam_<number>
    patterns = [
        r'H_cam_(\d+)',  # H_cam_234
        r'H(\d+)',        # H260
    ]

    for pattern in patterns:
        match = re.search(pattern, filename)
        if match:
            height = int(match.group(1))
            # Round to nearest 10 cm
            return round(height / 10) * 10

    return None

In [ ]:
def read_yolo_annotations(labels_dir: str, subsets: list[str]) -> pd.DataFrame:
    """
    Read YOLO annotations from multiple subsets and create a DataFrame.
    
    Args:
        labels_dir: Path to labels directory
        subsets: List of subset names (train, val, holdout, holdout_v2)
    
    Returns:
        DataFrame with columns: filename, x, y, w, h, pos_x, pos_y, cam_height
    """
    all_annotations = []
    skipped_files = []
    
    for subset in subsets:
        subset_path = os.path.join(labels_dir, subset)
        
        if not os.path.exists(subset_path):
            print(f"Warning: {subset_path} does not exist, skipping...")
            continue
        
        txt_files = list(Path(subset_path).glob("*.txt"))
        print(f"Processing {subset}: {len(txt_files)} files")
        
        for txt_file in tqdm(txt_files, desc=f"Reading {subset}"):
            filename = txt_file.name
            
            # Extract camera height
            cam_height = extract_camera_height(filename)
            
            if cam_height is None:
                skipped_files.append(str(txt_file))
                continue
            
            # Read YOLO annotations from file
            try:
                with open(txt_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        # YOLO format: class_id x_center y_center width height
                        parts = line.split()
                        if len(parts) < 5:
                            continue
                        
                        class_id, x, y, w, h = parts[:5]
                        x, y, w, h = float(x), float(y), float(w), float(h)
                        
                        # Calculate position (top-left corner)
                        pos_x = x - w / 2
                        pos_y = y - h / 2
                        
                        all_annotations.append({
                            'filename': filename,
                            'x': x,
                            'y': y,
                            'w': w,
                            'h': h,
                            'pos_x': pos_x,
                            'pos_y': pos_y,
                            'cam_height': cam_height,
                            'subset': subset
                        })
            except Exception as e:
                print(f"Error reading {txt_file}: {e}")
    
    print(f"\nSkipped {len(skipped_files)} files without camera height information")
    
    return pd.DataFrame(all_annotations)

In [ ]:
# Test camera height extraction and rounding
test_filenames = [
    "4.2_batch1____4.2_H260_menivim_5.txt",
    "4.2_batch1____4.2_H277_menivim_OS_28.txt",
    "4.2_batch2____4.2_H_cam_234_spacesbooth_100.txt",
    "4.2_background____4.2_H_cam_180__5.txt",
    "4.2_H289_test.txt",
    "4.2_H272_test.txt"
]

print("Testing camera height extraction (with rounding to nearest 10 cm):")
for filename in test_filenames:
    height = extract_camera_height(filename)
    print(f"{filename} -> {height} cm")

In [ ]:
# Read all annotations
df = read_yolo_annotations(DATASET_PATH, SUBSETS)

print(f"\nTotal annotations: {len(df)}")
print(f"Unique files: {df['filename'].nunique()}")
print(f"\nDataset splits:")
print(df['subset'].value_counts())

In [ ]:
# Display sample data
df.head(10)

In [ ]:
# Check camera height distribution
print("Camera height distribution:")
print(df['cam_height'].value_counts().sort_index())

In [ ]:
df = df[df['cam_height'] >= 200]

In [ ]:
# Display DataFrame info
df.info()

In [ ]:
# Save to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nAnnotations saved to {OUTPUT_CSV}")
print(f"File size: {os.path.getsize(OUTPUT_CSV) / 1024 / 1024:.2f} MB")

In [ ]:
# Summary statistics
print("\n=== Summary Statistics ===")
print(f"Total bounding boxes: {len(df)}")
print(f"\nBy subset:")
for subset in SUBSETS:
    count = len(df[df['subset'] == subset])
    files = df[df['subset'] == subset]['filename'].nunique()
    print(f"  {subset}: {count} boxes in {files} files")

print(f"\nCamera heights (cm):")
print(f"  Min: {df['cam_height'].min()}")
print(f"  Max: {df['cam_height'].max()}")
print(f"  Mean: {df['cam_height'].mean():.1f}")
print(f"  Unique values: {df['cam_height'].nunique()}")